# Xarray-Spatial Fire: Burn severity, rate of spread, and drought indexing

Wildfire analysis splits into two phases: predicting how a fire will behave before it starts, and assessing the damage after it's out. Xarray-spatial's fire module covers both, with functions for dNBR-based burn severity mapping, Rothermel spread modeling, Byram's fireline intensity, and the Keetch-Byram Drought Index.

### What you'll build

1. Generate a synthetic fire landscape with pre- and post-fire vegetation indices
2. Map burn damage with dNBR and relative dNBR
3. Classify burn severity into USGS 7-class categories
4. Model fireline intensity and flame length from fuel loads
5. Compute rate of spread across different fuel types using Rothermel's model
6. Track drought conditions with the Keetch-Byram Drought Index

![Fire analysis preview](images/fire_analysis_preview.png)

[dNBR](#dNBR) · [RdNBR](#RdNBR) · [Burn Severity Classification](#Burn-Severity-Classification) · [Fireline Intensity](#Fireline-Intensity) · [Flame Length](#Flame-Length) · [Rate of Spread](#Rate-of-Spread) · [KBDI](#KBDI)

Standard imports plus the fire submodule.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch

import xrspatial
from xrspatial.fire import (
    dnbr, rdnbr, burn_severity_class,
    fireline_intensity, flame_length,
    rate_of_spread, kbdi,
)

## Synthetic fire landscape

We build a 200x200 landscape with a simulated burn scar in the center. Pre-fire NBR is higher where vegetation is denser; after the fire, NBR drops inside the burn perimeter. We also create slope, wind, fuel moisture, and weather grids for the fire behavior and drought functions.

In a real workflow you would compute NBR from satellite imagery using `xrspatial.multispectral.nbr` (see User Guide 6).

In [ ]:
H, W = 200, 200
rng = np.random.default_rng(42)

# Coordinates
ys = np.linspace(H - 1, 0, H)
xs = np.linspace(0, W - 1, W)

def make_da(data, name):
    return xr.DataArray(data.astype(np.float32), dims=['y', 'x'],
                        coords={'y': ys, 'x': xs}, name=name)

# Pre-fire NBR: smooth vegetation gradient + noise (range roughly 0.1 to 0.7)
yy, xx = np.meshgrid(np.linspace(0, 1, H), np.linspace(0, 1, W), indexing='ij')
veg_gradient = 0.3 + 0.3 * np.sin(2 * np.pi * yy) * np.cos(np.pi * xx)
pre_nbr = veg_gradient + rng.normal(0, 0.03, (H, W))
pre_nbr = np.clip(pre_nbr, 0.05, 0.85)

# Burn scar: elliptical region in the center where NBR drops
cy, cx = H // 2, W // 2
dist = np.sqrt(((yy - 0.5) / 0.25) ** 2 + ((xx - 0.5) / 0.35) ** 2)
burn_mask = dist < 1.0
burn_intensity = np.clip(1.0 - dist, 0, 1)

post_nbr = pre_nbr.copy()
post_nbr[burn_mask] -= burn_intensity[burn_mask] * (0.3 + rng.uniform(0, 0.3, burn_mask.sum()))
post_nbr = np.clip(post_nbr, -0.5, 0.85)

pre_nbr_agg = make_da(pre_nbr, 'pre_nbr')
post_nbr_agg = make_da(post_nbr, 'post_nbr')

print(f"Pre-fire NBR range:  {pre_nbr.min():.3f} to {pre_nbr.max():.3f}")
print(f"Post-fire NBR range: {post_nbr.min():.3f} to {post_nbr.max():.3f}")

Pre-fire NBR is highest in the vegetation-dense zones (green). The burn scar in the center drops post-fire NBR, visible as a brown patch in the right panel.

In [ ]:
veg_cmap = LinearSegmentedColormap.from_list('veg', ['brown', 'yellow', 'green'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
pre_nbr_agg.plot.imshow(ax=axes[0], cmap=veg_cmap, add_colorbar=False)
axes[0].set_title('Pre-fire NBR', fontsize=13)
axes[0].set_axis_off()
post_nbr_agg.plot.imshow(ax=axes[1], cmap=veg_cmap, add_colorbar=False)
axes[1].set_title('Post-fire NBR', fontsize=13)
axes[1].set_axis_off()
plt.tight_layout()

## dNBR

The differenced Normalized Burn Ratio is the simplest burn severity metric: `pre_NBR - post_NBR`. Positive values indicate vegetation loss (burn damage); negative values indicate regrowth or increased greenness. The magnitude roughly corresponds to how much the fire changed the surface.

dNBR is the starting point for most burn severity analyses. USGS and BAER teams use it as input to the severity classification thresholds shown in the next section.

In [ ]:
dnbr_agg = dnbr(pre_nbr_agg, post_nbr_agg)

print(f"dNBR range: {float(dnbr_agg.min()):.3f} to {float(dnbr_agg.max()):.3f}")

fire_cmap = LinearSegmentedColormap.from_list(
    'fire', ['steelblue', 'lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
dnbr_agg.plot.imshow(ax=ax, cmap=fire_cmap, add_colorbar=True,
                     cbar_kwargs={'label': 'dNBR', 'shrink': 0.7})
ax.set_axis_off()

The burn scar is visible in the center. Dark red pixels had the largest drop in NBR (most severely burned). Blue areas outside the perimeter show slight increases, which could indicate seasonal greening or sensor noise. You can also call this through the `.xrs` accessor: `pre_nbr_agg.xrs.dnbr(post_nbr_agg)`.

## RdNBR

Relative dNBR normalizes the burn severity by the pre-fire vegetation density:

```
RdNBR = dNBR / sqrt(abs(pre_NBR / 1000))
```

This matters because a dNBR of 0.3 means something different in a dense forest (pre_NBR = 0.7) than in sparse grassland (pre_NBR = 0.2). RdNBR lets you compare severity across vegetation types on the same scale.

Pixels where pre-fire NBR is near zero get set to NaN to avoid dividing by a tiny number.

In [ ]:
rdnbr_agg = rdnbr(dnbr_agg, pre_nbr_agg)

print(f"RdNBR range: {float(np.nanmin(rdnbr_agg.data)):.3f} to "
      f"{float(np.nanmax(rdnbr_agg.data)):.3f}")

fig, ax = plt.subplots(figsize=(10, 7.5))
rdnbr_agg.plot.imshow(ax=ax, cmap=fire_cmap, add_colorbar=True,
                      cbar_kwargs={'label': 'RdNBR', 'shrink': 0.7})
ax.set_axis_off()

The spatial pattern looks similar to dNBR, but RdNBR rescales by pre-fire vegetation density. Sparse areas that burned can show higher RdNBR even when their raw dNBR was small.

<div class="alert alert-block alert-warning">
<b>dNBR vs. RdNBR.</b> RdNBR is better for comparing severity across different vegetation types, but it amplifies noise in sparse areas where pre-fire NBR is low. For uniform landscapes, plain dNBR is often enough.
</div>

## Burn Severity Classification

The `burn_severity_class` function bins dNBR values into the standard USGS 7-class scheme:

| Class | Label | dNBR range |
|-------|-------|------------|
| 1 | Enhanced regrowth (high) | < -0.251 |
| 2 | Enhanced regrowth (low) | -0.251 to -0.101 |
| 3 | Unburned | -0.101 to 0.099 |
| 4 | Low severity | 0.099 to 0.269 |
| 5 | Moderate-low severity | 0.269 to 0.439 |
| 6 | Moderate-high severity | 0.439 to 0.659 |
| 7 | High severity | >= 0.659 |

Output is int8. Class 0 means nodata (NaN input). This function has `@supports_dataset`, so you can pass an `xr.Dataset` and it will classify each variable.

In [ ]:
severity = burn_severity_class(dnbr_agg)

# Count pixels per class
labels = {
    1: 'Enhanced regrowth (high)',
    2: 'Enhanced regrowth (low)',
    3: 'Unburned',
    4: 'Low severity',
    5: 'Moderate-low',
    6: 'Moderate-high',
    7: 'High severity',
}
for cls, label in labels.items():
    n = int(np.sum(severity.data == cls))
    print(f"  Class {cls} ({label}): {n} pixels")

# Cast to float for plotting; map nodata (0) to NaN
severity_float = severity.astype(np.float32)
severity_float = xr.where(severity_float == 0, np.nan, severity_float)

severity_colors = ['#1a9641', '#6db856', '#a6d96a', '#fee08b',
                   '#fdae61', '#d73027', '#67001f']
severity_cmap = LinearSegmentedColormap.from_list('severity', severity_colors, N=7)

fig, ax = plt.subplots(figsize=(10, 7.5))
severity_float.plot.imshow(ax=ax, cmap=severity_cmap, vmin=1, vmax=7,
                           add_colorbar=False)
ax.legend(handles=[
    Patch(facecolor=severity_colors[6], alpha=0.78, label='High severity'),
    Patch(facecolor=severity_colors[4], alpha=0.78, label='Moderate-low'),
    Patch(facecolor=severity_colors[2], alpha=0.78, label='Unburned'),
    Patch(facecolor=severity_colors[0], alpha=0.78, label='Enhanced regrowth'),
], loc='lower right', fontsize=11, framealpha=0.9)
ax.set_axis_off()

<div class="alert alert-block alert-warning">
<b>Thresholds are guidelines, not law.</b> The USGS severity classes use fixed dNBR breakpoints that were calibrated for conifer forests in the western US. Other ecosystems (grasslands, boreal forest, tropical savanna) may need adjusted thresholds. Always validate with field data when possible.
</div>

## Fireline Intensity

[Byram's fireline intensity](https://www.frames.gov/catalog/1580) is the rate of heat release per unit length of fire front: `I = H * w * R`, where *H* is heat content (kJ/kg, default 18,000), *w* is fuel consumed per unit area (kg/m^2), and *R* is spread rate (m/s). Output is in kW/m.

Fireline intensity drives suppression difficulty. Fires below about 350 kW/m can be attacked by hand crews; above 4,000 kW/m they typically need indirect attack or aerial resources. The plot shows intensity as a continuous heat map.

In [ ]:
# Fuel consumed (kg/m^2): varies with vegetation density
fuel = (veg_gradient * 3.0 + rng.uniform(0, 0.5, (H, W))).astype(np.float32)
fuel_agg = make_da(fuel, 'fuel_consumed')

# Spread rate (m/s): moderate fire, variable across the grid
spread = (0.02 + 0.03 * rng.uniform(0, 1, (H, W))).astype(np.float32)
spread_agg = make_da(spread, 'spread_rate')

intensity_agg = fireline_intensity(fuel_agg, spread_agg, heat_content=18000)

print(f"Fireline intensity range: {float(intensity_agg.min()):.1f} to "
      f"{float(intensity_agg.max()):.1f} kW/m")

intensity_cmap = LinearSegmentedColormap.from_list(
    'intensity', ['lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
intensity_agg.plot.imshow(ax=ax, cmap=intensity_cmap, add_colorbar=True,
                          cbar_kwargs={'label': 'Fireline intensity (kW/m)', 'shrink': 0.7})
ax.set_axis_off()

## Flame Length

Flame length in meters, derived from fireline intensity using Byram's equation: `L = 0.0775 * I^0.46`. Negative or zero intensity yields zero flame length.

The plot maps expected flame length across the landscape. Like `burn_severity_class`, this function supports `@supports_dataset`.

In [ ]:
fl_agg = flame_length(intensity_agg)

print(f"Flame length range: {float(fl_agg.min()):.2f} to {float(fl_agg.max()):.2f} m")

flame_cmap = LinearSegmentedColormap.from_list('flame', ['lightyellow', 'orange', 'red'])

fig, ax = plt.subplots(figsize=(10, 7.5))
fl_agg.plot.imshow(ax=ax, cmap=flame_cmap, add_colorbar=True,
                   cbar_kwargs={'label': 'Flame length (m)', 'shrink': 0.7})
ax.set_axis_off()

## Rate of Spread

The [`rate_of_spread`](https://www.fs.usda.gov/treesearch/pubs/32533) function implements a simplified Rothermel (1972) spread model. It takes per-pixel slope (degrees), mid-flame wind speed (km/h), and dead fuel moisture (fraction 0-1), plus a scalar fuel model number from the [Anderson 13](https://www.fs.usda.gov/treesearch/pubs/6447) table.

The fuel model selects parameters that describe the fuel bed (loading, surface-area-to-volume ratio, bed depth, moisture of extinction). The function pre-computes the Rothermel constants from these and evaluates spread rate per pixel. The plot shows how spread varies with slope and wind for fuel model 1 (short grass).

In [ ]:
# Create spatially varying inputs
slope_data = (5.0 + 20.0 * yy).astype(np.float32)  # steeper toward the top
wind_data = (5.0 + 15.0 * xx).astype(np.float32)    # windier toward the right
moisture_data = np.full((H, W), 0.06, dtype=np.float32)  # 6% dead fuel moisture

slope_agg = make_da(slope_data, 'slope')
wind_agg = make_da(wind_data, 'wind_speed')
moisture_agg = make_da(moisture_data, 'fuel_moisture')

ros_agg = rate_of_spread(slope_agg, wind_agg, moisture_agg, fuel_model=1)

print(f"Rate of spread range: {float(ros_agg.min()):.2f} to "
      f"{float(ros_agg.max()):.2f} m/min")

ros_cmap = LinearSegmentedColormap.from_list('ros', ['lightyellow', 'orange', 'red', 'darkred'])

fig, ax = plt.subplots(figsize=(10, 7.5))
ros_agg.plot.imshow(ax=ax, cmap=ros_cmap, add_colorbar=True,
                    cbar_kwargs={'label': 'Rate of spread (m/min)', 'shrink': 0.7})
ax.set_axis_off()

Spread rate is highest in the top-right corner, where slope and wind are both strongest. Fire climbs steep slopes faster, and wind pushes the flame front forward.

### Comparing fuel models

Different fuel types spread fire at very different rates. Below, we compare a few models with the same slope, wind, and moisture.

In [11]:
models_to_compare = [1, 3, 4, 8]
model_names = {1: 'Short grass', 3: 'Tall grass', 4: 'Chaparral', 8: 'Timber litter'}

for fm in models_to_compare:
    r = rate_of_spread(slope_agg, wind_agg, moisture_agg, fuel_model=fm)
    print(f"  Model {fm:2d} ({model_names[fm]:15s}): "
          f"{float(r.min()):8.2f} to {float(r.max()):8.2f} m/min")

  Model  1 (Short grass    ):    50.30 to   705.01 m/min
  Model  3 (Tall grass     ):    34.68 to   519.83 m/min
  Model  4 (Chaparral      ):   109.11 to  1551.39 m/min
  Model  8 (Timber litter  ):     4.46 to    58.96 m/min


### Effect of fuel moisture

Fuel moisture is the biggest factor working against fire spread. As dead fuel moisture approaches the moisture of extinction, reaction intensity drops and spread rate falls off sharply.

In [12]:
# Fixed slope and wind, vary moisture
flat = make_da(np.full((H, W), 10.0, dtype=np.float32), 'slope')
wind10 = make_da(np.full((H, W), 10.0, dtype=np.float32), 'wind')

moistures = [0.03, 0.06, 0.08, 0.10, 0.12]
for m in moistures:
    m_agg = make_da(np.full((H, W), m, dtype=np.float32), 'moisture')
    r = rate_of_spread(flat, wind10, m_agg, fuel_model=1)
    print(f"  Moisture {m:.0%}: {float(r.mean()):8.2f} m/min")

  Moisture 3%:   135.57 m/min
  Moisture 6%:   106.63 m/min
  Moisture 8%:    92.09 m/min
  Moisture 10%:    60.86 m/min
  Moisture 12%:     0.00 m/min


<div class="alert alert-block alert-info">
<b>Anderson 13 vs. Scott/Burgan 40.</b> The Anderson 13 fuel models are the classic set, but the <a href="https://www.fs.usda.gov/treesearch/pubs/9521">Scott and Burgan (2005) 40-model system</a> splits fuel into live and dead components with finer granularity. If you need those, build a custom fuel parameter array and feed it into the Rothermel equations directly.
</div>

## KBDI

The [Keetch-Byram Drought Index](https://www.srs.fs.usda.gov/pubs/rp/rp_se038.pdf) tracks cumulative soil moisture deficit on a 0 to 800 scale. It updates daily from maximum temperature and precipitation. Precipitation reduces the deficit (after subtracting 5.08 mm for canopy interception), while warm days increase it through evapotranspiration.

Fire weather forecasters use KBDI routinely. Values above 600 indicate extreme drought where organic soils can ignite. The `kbdi` function computes a single time-step update. To track conditions over a season, call it in a loop, feeding each day's output as the next day's `kbdi_prev`. The plot shows spatial variation after one hot, dry day.

In [ ]:
# Start from moderate drought (KBDI = 300), hot dry day
kbdi_prev = make_da(np.full((H, W), 300.0, dtype=np.float32), 'kbdi_prev')
max_temp = make_da((30.0 + 5.0 * yy).astype(np.float32), 'max_temp')  # warmer in north
precip = make_da(np.zeros((H, W), dtype=np.float32), 'precip')  # no rain

kbdi_day1 = kbdi(kbdi_prev, max_temp, precip, annual_precip=1200.0)

print(f"KBDI after day 1: {float(kbdi_day1.min()):.1f} to {float(kbdi_day1.max()):.1f}")

drought_cmap = LinearSegmentedColormap.from_list(
    'drought', ['#ffffcc', '#fed976', '#fd8d3c', '#bd0026'])

fig, ax = plt.subplots(figsize=(10, 7.5))
kbdi_day1.plot.imshow(ax=ax, cmap=drought_cmap, add_colorbar=True,
                      cbar_kwargs={'label': 'KBDI', 'shrink': 0.7})
ax.set_axis_off()

### Multi-day accumulation

Here we run KBDI forward for 30 hot, dry days, then simulate a 40 mm rain event.

In [14]:
# 30 days of hot, dry weather
current_kbdi = kbdi_prev.copy()
no_rain = make_da(np.zeros((H, W), dtype=np.float32), 'precip')
hot_day = make_da(np.full((H, W), 35.0, dtype=np.float32), 'temp')

history = [float(current_kbdi.mean())]
for day in range(30):
    current_kbdi = kbdi(current_kbdi, hot_day, no_rain, annual_precip=1200.0)
    history.append(float(current_kbdi.mean()))

# Day 31: 40 mm rain event
rain = make_da(np.full((H, W), 40.0, dtype=np.float32), 'precip')
current_kbdi = kbdi(current_kbdi, hot_day, rain, annual_precip=1200.0)
history.append(float(current_kbdi.mean()))

# A few more dry days after the rain
for day in range(5):
    current_kbdi = kbdi(current_kbdi, hot_day, no_rain, annual_precip=1200.0)
    history.append(float(current_kbdi.mean()))

print("Day  0:", f"{history[0]:.1f}")
print("Day 15:", f"{history[15]:.1f}")
print("Day 30:", f"{history[30]:.1f} (pre-rain)")
print("Day 31:", f"{history[31]:.1f} (post-rain)")
print("Day 36:", f"{history[-1]:.1f} (5 days after rain)")

Day  0: 300.0
Day 15: 677.8
Day 30: 770.1 (pre-rain)
Day 31: 741.0 (post-rain)
Day 36: 763.1 (5 days after rain)


The drought index climbs steadily during the dry spell, drops when 40 mm of rain falls (minus 5.08 mm canopy interception), then starts climbing again.

<div class="alert alert-block alert-warning">
<b>Temperature units.</b> The <code>kbdi</code> function expects maximum temperature in degrees Celsius and precipitation in millimeters. Passing Fahrenheit or inches will produce silently wrong results. Check your input units before running.
</div>

### References

- [USGS Burn Severity Portal](https://burnseverity.cr.usgs.gov/)
- [FIREMON Landscape Assessment](https://www.fs.usda.gov/treesearch/pubs/24042) (Key and Benson, 2006)
- [Relative dNBR](https://doi.org/10.1016/j.rse.2007.04.003) (Miller and Thode, 2007)
- [Rothermel's fire spread model](https://www.fs.usda.gov/treesearch/pubs/32533) (Rothermel, 1972)
- [Anderson 13 fuel models](https://www.fs.usda.gov/treesearch/pubs/6447) (Anderson, 1982)
- [Keetch-Byram Drought Index](https://www.srs.fs.usda.gov/pubs/rp/rp_se038.pdf) (Keetch and Byram, 1968)
- [Byram's fireline intensity](https://www.frames.gov/catalog/1580) (Byram, 1959)